# Distillation: DenseNet-121 taught by the DenseNet-121 + BioViL-T average (NIH official split, seed 42)

In [1]:
# ============================ CONFIG =========================== =
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import json, glob, time, math, random, hashlib, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
warnings.filterwarnings("ignore")

RUN_TRAINING = True
RUN_TEACHER = False
RUN_STUDENT = False
DEBUG = False    
SEED = 42               

PROTOCOL_VERSION = "2.2.0"
OUT_DIR   = os.environ.get("CXR_OUT_DIR", "/kaggle/working")
INPUT_DIR = os.environ.get("CXR_INPUT_DIR", "/kaggle/input")
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"      
NUM_WORKERS = 4

DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
VAL_FRACTION, SPLIT_SEED = 0.15, 42                              
EXPECTED_COUNTS = {"train_val": 86524, "test": 25596, "total": 112120}

CLASSES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
           "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
C = len(CLASSES)

ASL = dict(gamma_neg=4, gamma_pos=1, clip=0.05)
CNN_CFG = dict(
    kind="cnn", model_id="M08_DENSENET121_ASL_384_CR", timm_name="densenet121",
    img_size=384, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
    batch_size=32, warmup_epochs=1, finetune_epochs=8, patience=2,
    lr_head_warmup=1e-3, lr_head=1e-4, lr_backbone=1e-4,
    weight_decay=0.05, drop_path=0.1, use_ema=False, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=8.0
)

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Data and official split

In [2]:
from sklearn.model_selection import GroupShuffleSplit
meta_csv = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
if not os.path.exists(meta_csv):
    meta_csv = next((os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR) for f in fs if f == "Data_Entry_2017.csv"), meta_csv)
assert os.path.exists(meta_csv), "Data_Entry_2017.csv not found - check DATA_ROOT"
DATA_ROOT = os.path.dirname(meta_csv)
png = glob.glob(os.path.join(DATA_ROOT, "images_*", "images", "*.png")) or \
      [os.path.join(r, f) for r, _, fs in os.walk(DATA_ROOT) for f in fs if f.endswith(".png")]
df = pd.read_csv(meta_csv)
df["path"] = df["Image Index"].map({os.path.basename(p): p for p in png})
assert len(df) == EXPECTED_COUNTS["total"] and df.path.notna().all(), "images missing on disk"
labels = df["Finding Labels"].str.split("|")
for c in CLASSES:
    df[c] = labels.apply(lambda L, c=c: c in L).astype(np.float32)
df = df.sort_values("Image Index").reset_index(drop=True)

def read_list(fname):
    p = os.path.join(DATA_ROOT, fname)
    return open(p).read().split()
tv_ids, te_ids = read_list("train_val_list.txt"), read_list("test_list.txt")

df["split"] = np.where(df["Image Index"].isin(set(te_ids)), "test", "trainval")
tv_frame = df[df.split == "trainval"]
_, va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SPLIT_SEED)
             .split(tv_frame, groups=tv_frame["Patient ID"]))
df.loc[tv_frame.index, "split"] = "train"
df.loc[tv_frame.index[va], "split"] = "val"

df_train, df_val, df_test = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))

## 2. Model and training code

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_determinism():
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

class CXRDataset(Dataset):
    def __init__(self, paths, y, cfg, train):
        self.paths, self.y, s = paths, y.astype(np.float32), cfg["img_size"]
        self.tfm = (T.Compose([T.RandomResizedCrop(s, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5),
                               T.RandomApply([T.RandomRotation(7)], p=0.5), T.ColorJitter(brightness=0.1, contrast=0.1),
                               T.ToTensor()]) if train else
                    T.Compose([T.Resize((s, s), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()]))
        self.norm = T.Normalize(cfg["mean"], cfg["std"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")
        x = self.tfm(img).repeat(3, 1, 1)
        if self.norm is not None:
            x = self.norm(x)
        return x, self.y[i]

def loader(frame, cfg, train, batch_size=None, y=None):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(CXRDataset(frame["path"].values, frame[CLASSES].values if y is None else y, cfg, train),
                      batch_size=batch_size or cfg["batch_size"], shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=worker_init_fn, generator=g)
    
def run_folder_name(model_id):
    return f"{DATASET_ID}__{SPLIT_ID}__{model_id}__s{SEED}"
    
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p
        loss_pos = y * torch.log(p.clamp(min=self.eps)) * (1 - p).pow(self.gp)
        loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=self.eps)) * p_neg.pow(self.gn)
        return -(loss_pos + loss_neg).sum(1).mean()

class CNNClassifier(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.backbone.num_features, C))

    def forward(self, x):
        return self.head(self.backbone(x))

    def backbone_param_groups(self, cfg):
        return [{"params": list(self.backbone.parameters()), "lr": cfg["lr_backbone"]}]

def build_model(cfg, pretrained=True):
    seed_everything(SEED)
    model = CNNClassifier(cfg, pretrained)
    return model.to(DEVICE)

@torch.no_grad()
def predict_logits(model, dl):
    model.eval()
    out = []
    for x, _ in dl:
        with autocast():
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy().astype(np.float32)

def configure_phase(model, cfg, phase):
    for p in model.parameters():
        p.requires_grad = False
    groups = [{"params": list(model.head.parameters()), "lr": cfg["lr_head_warmup"] if phase == "warmup" else cfg["lr_head"]}]
    if phase == "finetune":
        groups += model.backbone_param_groups(cfg)
    for g in groups:
        for p in g["params"]:
            p.requires_grad = True
    return torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

def cosine_schedule(opt, total_steps, warmup_frac=0.03):
    w = max(1, int(total_steps * warmup_frac))
    f = lambda s: (s + 1) / w if s < w else 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, total_steps - w)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def val_macro_auroc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(np.mean([roc_auc_score(y[:, c], s[:, c]) for c in range(C) if 0 < y[:, c].sum() < len(y)]))

def train_model(cfg, exp_dir, model, y_train=None):
    loss_fn = AsymmetricLoss(**cfg["asl"])
    train_dl, val_dl = loader(df_train, cfg, True, y=y_train), loader(df_val, cfg, False, cfg["batch_size"] * 2)
    yv = df_val[CLASSES].values
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    plan = [("warmup", i) for i in range(cfg["warmup_epochs"])] + [("finetune", i) for i in range(cfg["finetune_epochs"])]
    best, best_epoch, bad, phase_now, t0, ep_times = -1.0, 0, 0, None, time.time(), []
    for ep, (phase, _) in enumerate(plan, 1):
        if phase != phase_now:
            phase_now, bad = phase, 0
            opt = configure_phase(model, cfg, phase)
            n_ep = cfg["warmup_epochs"] if phase == "warmup" else cfg["finetune_epochs"]
            sched = cosine_schedule(opt, n_ep * len(train_dl))
        model.train()
        run_loss = 0.0
        for i, (x, y) in enumerate(train_dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item()
        zv = predict_logits(model, val_dl)
        auc = val_macro_auroc(yv, zv)
        ep_times.append(time.time() - t0)
        print(f"Epoch {ep} [{phase}] loss {run_loss / len(train_dl):.4f} | val macro AUROC {auc:.4f}", flush=True)
        if auc > best:
            best, best_epoch = auc, ep
            torch.save(model.state_dict(), f"{exp_dir}/checkpoint_best.pt")
    model.load_state_dict(torch.load(f"{exp_dir}/checkpoint_best.pt", map_location=DEVICE))
    return model, dict(best_val_macro_auroc=best, best_epoch=best_epoch, epochs_trained=len(ep_times))

## 5. Session D3: Execution & Training

In [4]:
if RUN_TRAINING:
    assert USE_AMP, "Training needs a GPU session"
    exp_dir = f"{OUT_DIR}/{run_folder_name(CNN_CFG['model_id'])}"
    os.makedirs(exp_dir, exist_ok=True)
    
    set_determinism()
    model = build_model(CNN_CFG)
    model, info = train_model(CNN_CFG, exp_dir, model, y_train=None)
    
    for part, frame in (("val", df_val), ("test", df_test)):
        z = predict_logits(model, loader(frame, CNN_CFG, False, CNN_CFG["batch_size"] * 2))
        np.savez(f"{exp_dir}/logits_{part}.npz", image_index=frame["Image Index"].to_numpy(dtype=str),
                 patient_id=frame["Patient ID"].to_numpy(dtype=np.int64), logits=z,
                 labels=frame[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 split_id=SPLIT_ID, split_sha256="", model_id=CNN_CFG["model_id"], seed=SEED, eval_set=part)
                 
    import timm, transformers
    with open(f"{exp_dir}/config.json", "w") as f:
        json.dump(dict(
            model_id=CNN_CFG["model_id"], seed=SEED, split_id=SPLIT_ID, protocol_version=PROTOCOL_VERSION,
            model_cfg=CNN_CFG, train_info=info, software=dict(torch=torch.__version__, timm=timm.__version__)
        ), f, indent=2)
    print(f"Standalone {CNN_CFG['model_id']} training completed successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B / 32.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 1 [warmup] loss 0.3843 | val macro AUROC 0.7377
Epoch 2 [finetune] loss 0.3470 | val macro AUROC 0.8113
Epoch 3 [finetune] loss 0.3233 | val macro AUROC 0.8296
Epoch 4 [finetune] loss 0.3127 | val macro AUROC 0.8460
Epoch 5 [finetune] loss 0.3034 | val macro AUROC 0.8456
Epoch 6 [finetune] loss 0.2933 | val macro AUROC 0.8492
Epoch 7 [finetune] loss 0.2835 | val macro AUROC 0.8484
Epoch 8 [finetune] loss 0.2754 | val macro AUROC 0.8482
Epoch 9 [finetune] loss 0.2715 | val macro AUROC 0.8478
Standalone M08_DENSENET121_ASL_384_CR training completed successfully!
